# W06A — Diseño relacional mínimo: **PK/FK**, llaves surrogate y “star schema” (DuckDB)

**Objetivo:** pasar de “join por hostname” a un modelo **fact/dim con llaves estables**:
- **PK** (Primary Key) para identificar filas,
- **FK** (Foreign Key) para asegurar **integridad referencial**,
- **Surrogate keys** para dimensiones (ej. `host_id`),
- Validación con **evidencia** (anti-join de huérfanos).

> Importante: en *analítica/warehouse* a veces no se “enfuerzan” constraints en producción.
> Aquí los usamos como **contrato ejecutable** + herramienta pedagógica.

## Bibliografía (W06A)

### DDIA (Kleppmann)
- **Cap. 2 — Data Models and Query Languages**
  - Relacional vs documento vs grafo (por qué SQL es útil cuando hay relaciones y joins).
  - Identificadores y relaciones (equivalentes a FK/referencias).

### Dimensional modeling (complementario)
- Kimball (surrogate keys en dimensiones y por qué ayudan en fact tables).

### DuckDB (práctica)
- Constraints: `PRIMARY KEY`, `FOREIGN KEY`, `UNIQUE`, `NOT NULL`

In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path("/home/manuela-rios/Documentos/Física computacional/Manuela-Rios")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
ART_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = RAW_DIR / "pscomppars.csv"
if not raw_csv.exists():
    raise FileNotFoundError(f"No encuentro {raw_csv}. Necesitas el CSV de W01/W02.")

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_quote(str(raw_csv.resolve()))})
""")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)

con.execute("DROP TABLE IF EXISTS silver_planet")
con.execute("""
CREATE TABLE silver_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  sy_snum,
  sy_pnum,
  sy_dist,
  ra,
  dec,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (disc_year IS NULL OR (disc_year BETWEEN 1980 AND 2026))
  AND (pl_rade IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR (pl_bmasse > 0))
""")

con.execute("DROP TABLE IF EXISTS dim_host_full")
con.execute("""
CREATE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(sy_dist) AS sy_dist,
  MAX(ra) AS ra,
  MAX(dec) AS dec,
  MAX(st_teff) AS st_teff,
  MAX(st_rad) AS st_rad,
  MAX(st_mass) AS st_mass
FROM silver_planet
GROUP BY hostname
""")

con.execute("DROP TABLE IF EXISTS fact_planet")
con.execute("""
CREATE TABLE fact_planet AS
SELECT DISTINCT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM silver_planet
""")

con.sql("SELECT COUNT(*) AS n_fact, COUNT(DISTINCT pl_name) AS n_pl, COUNT(DISTINCT hostname) AS n_hosts FROM fact_planet").show()


PROJECT_ROOT: /home/manuela-rios/Documentos/Física computacional/Manuela-Rios
DB_PATH: /home/manuela-rios/Documentos/Física computacional/Manuela-Rios/data/exoplanets.duckdb


## 1) Conceptos mínimos

### Grain (granularidad)
- `fact_planet`: **1 fila ~ 1 planeta** (identificado por `pl_name` en nuestro Core).
- `dim_host`: **1 fila ~ 1 estrella anfitriona** (identificada por `hostname`).

### Natural key vs Surrogate key
- **Natural key:** viene del dominio (ej. `hostname`).
- **Surrogate key:** entero generado (ej. `host_id`).
  - Ventajas típicas en warehouse: más pequeño, más estable para joins, más fácil para evolucionar modelos.

### PK/FK
- **PK**: “esta columna identifica una fila”.
- **FK**: “esta columna debe existir en la tabla padre”.

In [4]:
# TODO 1 (estudiante): crea dim_host_sk igual que en clase
# - Debe tener host_id como PRIMARY KEY
# - hostname NOT NULL y UNIQUE
# - Inserta datos desde dim_host_full usando ROW_NUMBER()

con.execute("DROP TABLE IF EXISTS dim_host_sk")

con.execute("""
CREATE TABLE dim_host_sk (
    host_id INTEGER PRIMARY KEY,
    hostname VARCHAR NOT NULL UNIQUE,
    sy_dist DOUBLE,
    ra DOUBLE,
    dec DOUBLE,
    st_teff DOUBLE,
    st_rad DOUBLE,
    st_mass DOUBLE
)
""")

con.execute("""
INSERT INTO dim_host_sk
SELECT
    ROW_NUMBER() OVER (ORDER BY hostname) AS host_id,
    hostname,
    sy_dist,
    ra,
    dec,
    st_teff,
    st_rad,
    st_mass
FROM dim_host_full
""")

# Validación:
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT hostname) AS n_keys FROM dim_host_sk").show()


┌─────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name   │ column_type │  null   │   key   │ default │  extra  │
│     varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ pl_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ hostname        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ discoverymethod │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ disc_year       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_snum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_pnum         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ pl_orbper       │ DOUBLE      │ YES 

In [5]:
con.execute("DROP TABLE IF EXISTS dim_host_full")

con.execute("""
CREATE TABLE dim_host_full AS
SELECT
    hostname,
    MAX(sy_dist) AS sy_dist,
    MAX(ra) AS ra,
    MAX(dec) AS dec,
    MAX(st_teff) AS st_teff,
    MAX(st_rad) AS st_rad,
    MAX(st_mass) AS st_mass
FROM silver_planet
GROUP BY hostname
""")

con.sql("DESCRIBE dim_host_full").show()


┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ hostname    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sy_dist     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ ra          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ dec         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ st_teff     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ st_rad      │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ st_mass     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [6]:
# TODO 1 (estudiante): crea dim_host_sk igual que en clase
# - Debe tener host_id como PRIMARY KEY
# - hostname NOT NULL y UNIQUE
# - Inserta datos desde dim_host_full usando ROW_NUMBER()

con.execute("DROP TABLE IF EXISTS dim_host_sk")

con.execute("""
CREATE TABLE dim_host_sk (
    host_id INTEGER PRIMARY KEY,
    hostname VARCHAR NOT NULL UNIQUE,
    sy_dist DOUBLE,
    ra DOUBLE,
    dec DOUBLE,
    st_teff DOUBLE,
    st_rad DOUBLE,
    st_mass DOUBLE
)
""")

con.execute("""
INSERT INTO dim_host_sk
SELECT
    ROW_NUMBER() OVER (ORDER BY hostname) AS host_id,
    hostname,
    sy_dist,
    ra,
    dec,
    st_teff,
    st_rad,
    st_mass
FROM dim_host_full
""")

# Validación:
con.sql("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT hostname) AS n_keys FROM dim_host_sk").show()

┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4537 │   4537 │
└────────┴────────┘



In [ ]:
# TODO 2 (estudiante): crea fact_planet_sk con FK a dim_host_sk(host_id)
# - pl_name PRIMARY KEY
# - host_id NOT NULL REFERENCES dim_host_sk(host_id)
# - Inserta desde fact_planet JOIN dim_host_sk (por hostname)

con.execute("DROP TABLE IF EXISTS fact_planet_sk")

con.execute("""
CREATE TABLE fact_planet_sk (
    pl_name VARCHAR PRIMARY KEY,
    host_id INTEGER NOT NULL REFERENCES dim_host_sk(host_id),
    discoverymethod VARCHAR,
    disc_year INTEGER,
    pl_orbper DOUBLE,
    pl_rade DOUBLE,
    pl_bmasse DOUBLE,
    pl_eqt DOUBLE
)
""")

con.execute("""
INSERT INTO fact_planet_sk
SELECT
    f.pl_name,
    d.host_id,
    f.discoverymethod,
    f.disc_year,
    f.pl_orbper,
    f.pl_rade,
    f.pl_bmasse,
    f.pl_eqt
FROM fact_planet f
JOIN dim_host_sk d ON f.hostname = d.hostname
""")

con.sql("SELECT COUNT(*) AS n_fact_sk FROM fact_planet_sk").show()

# Check huérfanos
con.sql('''
SELECT COUNT(*) AS orphan_rows
FROM fact_planet_sk f
LEFT JOIN dim_host_sk d ON f.host_id = d.host_id
WHERE d.host_id IS NULL
''').show()

┌───────────┐
│ n_fact_sk │
│   int64   │
├───────────┤
│      6101 │
└───────────┘

┌─────────────┐
│ orphan_rows │
│    int64    │
├─────────────┤
│           0 │
└─────────────┘



In [ ]:
# TU TURNO 3: consulta analítica usando el modelo con llaves
# Objetivo: top 10 métodos de descubrimiento y n_planets

q = '''
SELECT
  discoverymethod,
  COUNT(*) AS n_planets
FROM fact_planet_sk
WHERE discoverymethod IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
LIMIT 10
'''
con.sql(q).show()

┌───────────────────────────────┬───────────┐
│        discoverymethod        │ n_planets │
│            varchar            │   int64   │
├───────────────────────────────┼───────────┤
│ Transit                       │      4500 │
│ Radial Velocity               │      1166 │
│ Microlensing                  │       266 │
│ Imaging                       │        87 │
│ Transit Timing Variations     │        39 │
│ Eclipse Timing Variations     │        17 │
│ Orbital Brightness Modulation │         9 │
│ Pulsar Timing                 │         8 │
│ Astrometry                    │         6 │
│ Pulsation Timing Variations   │         2 │
├───────────────────────────────┴───────────┤
│ 10 rows                         2 columns │
└───────────────────────────────────────────┘



## Data Contract (El proyecto debería tener al menos) (`docs/data_contract.md`)
Incluye explícitamente:
- Datasets: `silver_planet`, `dim_host_sk`, `fact_planet_sk`, `gold_by_*`
- Grain:
  - `dim_host_sk`: 1 fila por `hostname`
  - `fact_planet_sk`: 1 fila por `pl_name`
- Keys:
  - PK dim: `host_id`, UNIQUE: `hostname`
  - PK fact: `pl_name`
  - FK: `fact_planet_sk.host_id → dim_host_sk.host_id`
- Checks mínimos (con evidencia):
  - uniqueness dim: `COUNT(*) == COUNT(DISTINCT hostname)`
  - orphans: `orphan_rows == 0`
  - `orphan_rows = 0`
  - `n_rows(dim) == n_keys(hostname)`

# Reinicio limpio del modelo para evitar arrastrar versiones viejas de tablas
con.execute("DROP VIEW IF EXISTS gold_by_discoverymethod")
con.execute("DROP VIEW IF EXISTS gold_by_host")
con.execute("DROP TABLE IF EXISTS fact_planet_sk")
con.execute("DROP TABLE IF EXISTS dim_host_sk")
con.execute("DROP TABLE IF EXISTS fact_planet")
con.execute("DROP TABLE IF EXISTS dim_host_full")
con.execute("DROP TABLE IF EXISTS silver_planet")


from pathlib import Path
import duckdb

PROJECT_ROOT = Path("/home/manuela-rios/Documentos/Física computacional/Manuela-Rios")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
ART_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = RAW_DIR / "pscomppars.csv"
if not raw_csv.exists():
    raise FileNotFoundError(f"No encuentro {raw_csv}. Necesitas el CSV de W01/W02.")

def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_quote(str(raw_csv.resolve()))})
""")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)

con.execute("DROP TABLE IF EXISTS silver_planet")
con.execute("""
CREATE TABLE silver_planet AS
SELECT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  sy_snum,
  sy_pnum,
  sy_dist,
  ra,
  dec,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (disc_year IS NULL OR (disc_year BETWEEN 1980 AND 2026))
  AND (pl_rade IS NULL OR (pl_rade > 0 AND pl_rade <= 30))
  AND (pl_bmasse IS NULL OR (pl_bmasse > 0))
""")

con.execute("DROP TABLE IF EXISTS dim_host_full")
con.execute("""
CREATE TABLE dim_host_full AS
SELECT
  hostname,
  MAX(sy_dist) AS sy_dist,
  MAX(ra) AS ra,
  MAX(dec) AS dec,
  MAX(st_teff) AS st_teff,
  MAX(st_rad) AS st_rad,
  MAX(st_mass) AS st_mass
FROM silver_planet
GROUP BY hostname
""")

con.execute("DROP TABLE IF EXISTS fact_planet")
con.execute("""
CREATE TABLE fact_planet AS
SELECT DISTINCT
  pl_name,
  hostname,
  discoverymethod,
  disc_year,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt
FROM silver_planet
""")

con.sql("SELECT COUNT(*) AS n_fact, COUNT(DISTINCT pl_name) AS n_pl, COUNT(DISTINCT hostname) AS n_hosts FROM fact_planet").show()


In [ ]:
# Eliminar primero la tabla con FK, luego la tabla padre
con.execute("DROP TABLE IF EXISTS fact_planet_sk")
con.execute("DROP TABLE IF EXISTS dim_host_sk")

In [ ]:
from pathlib import Path
import duckdb


def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'data').exists() and (p / 'notebooks').exists():
            return p
    return cur

PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
DB_PATH = PROJECT_ROOT / 'data' / 'exoplanets.duckdb'
ART_DIR = PROJECT_ROOT / 'artifacts'
DOCS_DIR = PROJECT_ROOT / 'docs'

RAW_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

raw_csv = RAW_DIR / 'pscomppars.csv'
if not raw_csv.exists():
    raise FileNotFoundError(f'No encuentro {raw_csv}. Necesitas el CSV de W01/W02.')

if not DB_PATH.parent.exists():
    raise FileNotFoundError(f'No existe el directorio de la base de datos: {DB_PATH.parent}')


def sql_quote(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"

con = duckdb.connect(str(DB_PATH))
con.execute(f"""
CREATE OR REPLACE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_quote(str(raw_csv.resolve()))})
""")

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DB_PATH:', DB_PATH)


┌────────┬────────┐
│ n_rows │ n_keys │
│ int64  │ int64  │
├────────┼────────┤
│   4550 │   4550 │
└────────┴────────┘

┌─────────────┐
│ orphan_rows │
│    int64    │
├─────────────┤
│           0 │
└─────────────┘

┌───────────┐
│ n_fact_sk │
│   int64   │
├───────────┤
│      6101 │
└───────────┘



## Tu Turno: construye 2 Gold outputs

In [ ]:
# TODO 1 (Gold 1): crea la vista gold_by_discoverymethod (igual que en clase)
# Requisitos:
# - COUNT(*) por discoverymethod
# - AVG(pl_rade) y/o AVG(pl_bmasse)
# - MIN(disc_year) / MAX(disc_year)
# - ORDER BY n_planets DESC

con.execute("DROP VIEW IF EXISTS gold_by_discoverymethod")

con.execute("""
CREATE VIEW gold_by_discoverymethod AS
SELECT
    discoverymethod,
    COUNT(*) AS n_planets,
    AVG(pl_rade) AS avg_radius_earth,
    AVG(pl_bmasse) AS avg_mass_jupiter,
    MIN(disc_year) AS first_detection,
    MAX(disc_year) AS last_detection
FROM fact_planet_sk
WHERE discoverymethod IS NOT NULL
GROUP BY discoverymethod
ORDER BY n_planets DESC
""")

con.sql("SELECT * FROM gold_by_discoverymethod LIMIT 10").show()

┌───────────────────────────────┬───────────┬────────────────────┬────────────────────┬─────────────────┬────────────────┐
│        discoverymethod        │ n_planets │  avg_radius_earth  │  avg_mass_jupiter  │ first_detection │ last_detection │
│            varchar            │   int64   │       double       │       double       │      int32      │     int32      │
├───────────────────────────────┼───────────┼────────────────────┼────────────────────┼─────────────────┼────────────────┤
│ Transit                       │      4500 │  4.361876653578584 │ 123.46020251712618 │            2002 │           2026 │
│ Radial Velocity               │      1166 │  9.759872661948648 │   1035.40510642866 │            1995 │           2026 │
│ Microlensing                  │       266 │  9.850563909774438 │  797.6943019686465 │            2004 │           2026 │
│ Imaging                       │        87 │  13.43342457831325 │  4485.179850102737 │            2004 │           2026 │
│ Transit Timing

In [ ]:
# TODO 3: exporta las 2 vistas Gold a CSV en artifacts/

out1 = ART_DIR / "gold_by_discoverymethod.csv"
out2 = ART_DIR / "gold_by_host.csv"

con.execute(f"COPY gold_by_discoverymethod TO '{out1}' (HEADER, DELIMITER ',')")
con.execute(f"COPY gold_by_host TO '{out2}' (HEADER, DELIMITER ',')")

print("Debe existir:", out1, "y", out2)


┌────────────┬───────────┬────────────────────┬────────────────────┐
│  hostname  │ n_planets │  avg_radius_earth  │  avg_mass_jupiter  │
│  varchar   │   int64   │       double       │       double       │
├────────────┼───────────┼────────────────────┼────────────────────┤
│ KOI-351    │         8 │                3.9 │           31.14875 │
│ TRAPPIST-1 │         7 │ 0.9785714285714285 │  0.921142857142857 │
│ TOI-1136   │         6 │  3.051819136666667 │               6.59 │
│ Kepler-20  │         6 │ 2.2926666666666664 │  9.386666666666667 │
│ HD 191939  │         6 │  6.590000000000001 │  176.9743020266667 │
│ HD 219134  │         6 │  3.668833333333333 │  25.23973666666667 │
│ HD 10180   │         6 │  6.676666666666667 │ 1587.7010169516664 │
│ TOI-178    │         6 │ 2.2176666666666667 │  4.051666666666667 │
│ K2-138     │         6 │ 2.5843333333333334 │  6.041666666666667 │
│ Kepler-80  │         6 │ 1.8133333333333332 │  747.3908333333333 │
├────────────┴───────────┴────────

## Tu Turno: export a artifacts/

In [ ]:
# TODO 3: exporta las 2 vistas Gold a CSV en artifacts/
# Ejecuta esta celda solo después de haber corrido las celdas donde se crean
# `gold_by_discoverymethod`, `gold_by_host` y la conexión `con`.

out1 = ART_DIR / 'gold_by_discoverymethod.csv'
out2 = ART_DIR / 'gold_by_host.csv'

con.execute(f"COPY gold_by_discoverymethod TO '{out1}' (HEADER, DELIMITER ',')")
con.execute(f"COPY gold_by_host TO '{out2}' (HEADER, DELIMITER ',')")

print('Debe existir:', out1, 'y', out2)


Debe existir: /home/santiago/Escritorio/UNIVERSIDAD/Castano-Paz-Bolivar/notebooks/artifacts/gold_by_discoverymethod.csv y /home/santiago/Escritorio/UNIVERSIDAD/Castano-Paz-Bolivar/notebooks/artifacts/gold_by_host.csv


### 


### 3) Decisions Log (`docs/decisions_log.md`)
Al final de W05 deben aparecer al menos 2 decisiones importantes:
- “Surrogate key + FK”
- “Gold outputs y por qué esas métricas”

## SOLUCIÓN ENTREGABLE

En esta versión quedó construido el modelo con surrogate key (`host_id`), la fact table con FK hacia la dimensión, dos salidas Gold y la exportación de evidencia a `artifacts/`.


In [ ]:
evidence_dim = con.execute('SELECT COUNT(*) AS n_rows, COUNT(DISTINCT hostname) AS n_keys FROM dim_host_sk').fetchall()
evidence_orphans = con.execute('''
SELECT COUNT(*) AS orphan_rows
FROM fact_planet_sk f
LEFT JOIN dim_host_sk d ON f.host_id = d.host_id
WHERE d.host_id IS NULL
''').fetchall()

print('dim_host_sk:', evidence_dim)
print('orphan_rows:', evidence_orphans)


In [ ]:
top_discovery = con.execute('SELECT * FROM gold_by_discoverymethod LIMIT 10').fetchall()
top_host = con.execute('SELECT * FROM gold_by_host LIMIT 10').fetchall()

print('Top 10 gold_by_discoverymethod')
for row in top_discovery:
    print(row)

print('\nTop 10 gold_by_host')
for row in top_host:
    print(row)
